# ASI07 Insecure Inter-Agent Communication — Upload Artifacts & Run Evaluation

**OWASP Category**: ASI07 — Insecure Inter-Agent Communication | **Risk Severity**: Critical

**Mapped LLM Categories**: LLM01, LLM06, LLM07

**ASI07 tests for**:
- Forged inter-agent messages (messages claiming false origin + escalated actions)
- Message interception and replay attacks (MITM, price tampering, authorization forgery)
- Implicit trust exploitation (channel-level trust abused with malicious payload injection)

In [ ]:
%pip install okareo python-dotenv --quiet

In [ ]:
import sys
from pathlib import Path
_nb = globals().get("__vsc_ipynb_file__", ".")
NOTEBOOK_DIR = Path(_nb).resolve().parent
CATEGORY_DIR = NOTEBOOK_DIR.parent
if str(CATEGORY_DIR.parent.parent) not in sys.path:
    sys.path.insert(0, str(CATEGORY_DIR.parent.parent))
from okareo.checks import ModelBasedCheck, CheckOutputType
from okareo.model_under_test import Driver
from owasp.common import init_okareo, parse_artifact, build_target, SINGLE_TURN_DRIVER_TEMPLATE
okareo, OKAREO_API_KEY = init_okareo()
print(f"✓ Okareo SDK initialized (key: ...{OKAREO_API_KEY[-5:]})")

In [ ]:
scenarios_dir = CATEGORY_DIR / "scenarios"
registered_scenarios = {}
for p in sorted(scenarios_dir.glob("*.jsonl")):
    name = f"ASI07-{p.stem}"
    sc = okareo.upload_scenario_set(scenario_name=name, file_path=str(p))
    registered_scenarios[name] = sc
    print(f"  ✓ {name}")

registered_checks = {}
for p in sorted((CATEGORY_DIR / "checks").glob("*.md")):
    data = parse_artifact(p)
    r = okareo.create_or_update_check(
        name=data["name"], description=data["description"],
        check=ModelBasedCheck(prompt_template=data["prompt_template"], check_type=CheckOutputType.PASS_FAIL),
    )
    registered_checks[data["name"]] = r.id
    print(f"  ✓ {data['name']}")

In [ ]:
target = build_target(CATEGORY_DIR)
IAC_CHECK = "ASI07-inter-agent-security-detector"
st_driver = Driver(name="pass-through", prompt_template=SINGLE_TURN_DRIVER_TEMPLATE, temperature=0)

all_results = {}
for sname, sc in registered_scenarios.items():
    try:
        run = okareo.run_simulation(
            target=target, driver=st_driver,
            name=f"ASI07 — {sname}",
            api_key=OKAREO_API_KEY, first_turn="target",
            scenario=sc, max_turns=1, checks=[IAC_CHECK],
        )
        all_results[sname] = run
        print(f"  ✓ {sname}: {getattr(run, 'app_link', run.id)}")
    except Exception as e:
        all_results[sname] = None; print(f"  ✗ {sname}: {e}")

errors = sum(1 for r in all_results.values() if r is None)
print(f"\nASI07 Complete — Total: {len(all_results)} | Errors: {errors}")